# v3 rebuild — Stage 1 · clean dataset

Regenerates the **leakage-free** dataset used by every downstream stage. This is the
concrete fix of the reviewer-visible data defects in
`REJECTION_RESPONSE_PLAN.md`:

* **D2** — drop 16 metadata-leak columns (`ID__*`, `Repetition__*`).
* **D3** — drop 96 byte-identical duplicated EEG columns
  (`eeg_features_5s__EEG_channel_*`, exact copies of the canonical `EEG_channel_*`).
* **D4** — keep `task_name` and the documented task→label rule; export the schema.

Identifier/session/task/window columns are retained **as keys only** so any
downstream split can be group-aware; they never enter the feature matrix.

Writes under `output/research_outputs/fusion_training/v3_rebuild/`:
`dataset_clean.parquet`, `feature_manifest.csv`, `task_label_schema.csv`,
`data_audit.json`. **Runtime: seconds.**


In [1]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd


def _root(start=Path.cwd()):
    for d in [start, *start.parents]:
        if (d / "src" / "rebuild" / "models_registry.py").exists():
            return d
    raise SystemExit("run notebook from the module root or src/")
REPO = _root()

FUSION = REPO / "output" / "research_outputs" / "fusion" / "v1_fusion" / "fusion_dataset.csv"
OUT = REPO / "output" / "research_outputs" / "fusion_training" / "v3_rebuild"
OUT.mkdir(parents=True, exist_ok=True)
print("fusion input :", FUSION, "->", FUSION.exists())
print("out dir      :", OUT)

fusion input : /home/g0amer/Desktop/thesis/01_Task_Recognition_Multimodal_Fusion/output/research_outputs/fusion/v1_fusion/fusion_dataset.csv -> True
out dir      : /home/g0amer/Desktop/thesis/01_Task_Recognition_Multimodal_Fusion/output/research_outputs/fusion_training/v3_rebuild


In [2]:
KEY_COLS_ALL = ["subject_id", "task_name", "task_file", "split", "window_idx",
                "start_idx", "end_idx", "n_samples", "label", "pseudo_label"]


def is_id_repetition_leak(c: str) -> bool:
    return c.split("__", 1)[0] in {"ID", "Repetition"}


DUPE_PREFIX = "eeg_features_5s__EEG_channel_"


def task_rule_label(task_name: str) -> str:
    name = str(task_name).lower()
    if any(x in name for x in ["stroop", "n-back", "mat", "hanoi"]):
        return "cognitive_load"
    if any(x in name for x in ["vr-plank"]):
        return "high_stress"
    if any(x in name for x in ["manual", "cobot"]):
        return "industrial_task"
    if any(x in name for x in ["rest", "meditation"]):
        return "low_load"
    return "other"


MODALITY = {
    "EEG": lambda c: re.match(r"EEG_channel_\d+__", c) is not None,
    "ECG": lambda c: c.startswith("ECG__"),
    "EDA": lambda c: c.startswith("EDA__"),
    "EMG": lambda c: c.startswith("EMG__"),
    "RESP": lambda c: c.startswith("RESP__"),
}


def modality_of(c: str) -> str:
    for name, fn in MODALITY.items():
        if fn(c):
            return name
    return "other"


df = pd.read_csv(FUSION, low_memory=False)
KEY_COLS = [c for c in KEY_COLS_ALL if c in df.columns]
print("loaded", df.shape, "| key cols kept as keys:", KEY_COLS)

loaded (5640, 257) | key cols kept as keys: ['subject_id', 'task_name', 'task_file', 'split', 'window_idx', 'start_idx', 'end_idx', 'n_samples', 'pseudo_label']


### Column manifest — what is kept / dropped and why

In [3]:
rows = []
for c in df.columns:
    if c in KEY_COLS:
        rows.append({"column": c, "group": "meta", "decision": "keep_as_key",
                     "reason": "identifier / label key"})
    elif is_id_repetition_leak(c):
        rows.append({"column": c, "group": "leak", "decision": "drop",
                     "reason": "recording ID/trial-counter metadata (D2)"})
    elif c.startswith(DUPE_PREFIX):
        short = c[len("eeg_features_5s__"):]
        rows.append({"column": c, "group": "EEG", "decision": "drop_duplicate",
                     "reason": f"byte-identical duplicate of {short} (D3)"})
    else:
        g = modality_of(c)
        if g == "other":
            rows.append({"column": c, "group": "other", "decision": "drop",
                         "reason": "dataset metadata / presence flag, not a physiological channel"})
        else:
            rows.append({"column": c, "group": g, "decision": "keep",
                         "reason": "physiological feature"})
manifest = pd.DataFrame(rows)
manifest.to_csv(OUT / "feature_manifest.csv", index=False)
manifest.decision.value_counts().rename_axis("decision").to_frame("columns")

,columns
decision,
keep,128
drop_duplicate,96
drop,24
keep_as_key,9


In [4]:
keep_cols = [r["column"] for r in rows if r["decision"] == "keep"]
group_of = {r["column"]: r["group"] for r in rows if r["decision"] == "keep"}
print("kept features:", len(keep_cols), "of", df.shape[1], "source cols")

X = df[keep_cols].apply(pd.to_numeric, errors="coerce")
missing = int(X.isna().sum().sum())
n_rows, n_feats = X.shape
print(f"missing feature cells: {missing}  ({100*missing/(n_rows*n_feats):.4f}%)")

(pd.Series([group_of[c] for c in keep_cols]).value_counts().rename_axis("group")
 .to_frame("n_features"))

kept features: 128 of 257 source cols
missing feature cells: 328  (0.0454%)


,n_features
group,
EEG,96
ECG,8
EDA,8
EMG,8
RESP,8


### Clean frame + label-schema export

In [5]:
clean = pd.concat([df[KEY_COLS], X], axis=1)
clean.to_parquet(OUT / "dataset_clean.parquet", index=False)

schema = (df[["subject_id", "task_name", "pseudo_label"]]
            .drop_duplicates("task_name")[["task_name", "pseudo_label"]]
            .rename(columns={"pseudo_label": "label_by_rule"})
            .sort_values(["label_by_rule", "task_name"]))
schema.to_csv(OUT / "task_label_schema.csv", index=False)
print("wrote dataset_clean.parquet:", clean.shape)
print("wrote task_label_schema.csv:", schema.shape)
schema

wrote dataset_clean.parquet: (5640, 137)
wrote task_label_schema.csv: (21, 2)


,task_name,label_by_rule
56,hanoi_0,cognitive_load
115,mat_0,cognitive_load
132,n-back_0,cognitive_load
153,stroopeasy_0,cognitive_load
156,stroophard_0,cognitive_load
158,vr-plank_0,high_stress
0,cobot-task-1,industrial_task
12,cobot-task-2,industrial_task
32,cobot-task-3,industrial_task
40,cobot-task-4,industrial_task


### Audit JSON + class balance

In [6]:
group_counts = pd.Series([group_of[c] for c in keep_cols]).value_counts()
audit = {
    "source_rows": int(len(df)),
    "source_cols": int(df.shape[1]),
    "key_cols": KEY_COLS,
    "dropped_leak_cols": int(((manifest.decision == "drop") & (manifest.group == "leak")).sum()),
    "dropped_dup_eeg_cols": int((manifest.decision == "drop_duplicate").sum()),
    "kept_feature_cols": int(n_feats),
    "kept_feature_cols_by_group": {k: int(v) for k, v in sorted(group_counts.items())},
    "missing_values_in_features": missing,
    "rows": int(n_rows),
    "subjects": int(df.subject_id.nunique()),
    "tasks": int(df.task_name.nunique()),
    "class_counts": {k: int(v) for k, v in df.pseudo_label.value_counts().items()},
    "outputs": {
        "dataset_clean": str(OUT / "dataset_clean.parquet"),
        "feature_manifest": str(OUT / "feature_manifest.csv"),
        "task_label_schema": str(OUT / "task_label_schema.csv"),
    },
}
(OUT / "data_audit.json").write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))

{
  "source_rows": 5640,
  "source_cols": 257,
  "key_cols": [
    "subject_id",
    "task_name",
    "task_file",
    "split",
    "window_idx",
    "start_idx",
    "end_idx",
    "n_samples",
    "pseudo_label"
  ],
  "dropped_leak_cols": 16,
  "dropped_dup_eeg_cols": 96,
  "kept_feature_cols": 128,
  "kept_feature_cols_by_group": {
    "ECG": 8,
    "EDA": 8,
    "EEG": 96,
    "EMG": 8,
    "RESP": 8
  },
  "missing_values_in_features": 328,
  "rows": 5640,
  "subjects": 52,
  "tasks": 21,
  "class_counts": {
    "industrial_task": 3571,
    "cognitive_load": 975,
    "low_load": 799,
    "high_stress": 224,
    "other": 71
  },
  "outputs": {
    "dataset_clean": "/home/g0amer/Desktop/thesis/01_Task_Recognition_Multimodal_Fusion/output/research_outputs/fusion_training/v3_rebuild/dataset_clean.parquet",
    "feature_manifest": "/home/g0amer/Desktop/thesis/01_Task_Recognition_Multimodal_Fusion/output/research_outputs/fusion_training/v3_rebuild/feature_manifest.csv",
    "task_label

Re-running this notebook is idempotent: it overwrites the four outputs with
identical bytes. **Expected final state**: 128 kept features (96 EEG + 32 peripheral),
16 leak + 96 duplicate + other columns dropped, 328 missing feature cells, 52 subjects,
21 tasks, 5640 windows, class counts `industrial_task 3571 / cognitive_load 975 /
low_load 799 / high_stress 224 / other 71`.